In [2]:
# add appropriate libraries
import pandas as pd
import numpy as np

In [3]:
# Create Tables
participants = pd.DataFrame({
    "id": [101, 102, 103, 104, 105, 106],
    "age": [24, 31, 27, 19, 42, 35],
    "group": ["A", "B", "A", "B", "A", "B"]
})
scores = pd.DataFrame({
    "id": [101, 102, 103, 105, 106, 107],
    "score": [81, 94, 88, 91, 85, 79]
})

Which IDs occur in both tables?
101, 102, 103, 105, and 106 occur in both tables

Which ID occurs only in participants?
104 appears only in participants.

Which ID occurs only in scores?
107 only appears in scores.

What do you expect an inner join to contain?
I expcted an inner join to only include the IDs in both tables.

In [7]:
# perform the inner join
inner = participants.merge(
    scores,
    on="id",
    how="inner"
)

inner

,id,age,group,score
0,101,24,A,81
1,102,31,B,94
2,103,27,A,88
3,105,42,A,91
4,106,35,B,85


Which IDs were removed?
104 and 107 were removed.

Why?
They only appear in one table each.

Did the number of rows match your prediction?
Yes.

In [9]:
# create a left join
left = participants.merge(
    scores,
    on="id",
    how="left"
)
left

,id,age,group,score
0,101,24,A,81.0
1,102,31,B,94.0
2,103,27,A,88.0
3,104,19,B,NaN
4,105,42,A,91.0
5,106,35,B,85.0


Which table defines the resulting population?
The participants table determins the resulting population.

What happened to ID 104?
104 is now included in the merge.

Why is its score missing?
The ID has no score, so an N/A value was added in its place.

In [10]:
outer = participants.merge(
    scores,
    on="id",
    how="outer"
)
outer

,id,age,group,score
0,101,24.0,A,81.0
1,102,31.0,B,94.0
2,103,27.0,A,88.0
3,104,19.0,B,NaN
4,105,42.0,A,91.0
5,106,35.0,B,85.0
6,107,NaN,NaN,79.0


In [11]:
diagnostic = participants.merge(
    scores,
    on="id",
    how="outer",
    indicator=True
)
diagnostic

,id,age,group,score,_merge
0,101,24.0,A,81.0,both
1,102,31.0,B,94.0,both
2,103,27.0,A,88.0,both
3,104,19.0,B,NaN,left_only
4,105,42.0,A,91.0,both
5,106,35.0,B,85.0,both
6,107,NaN,NaN,79.0,right_only


In [12]:
diagnostic["_merge"].value_counts()

_merge
both          5
left_only     1
right_only    1
Name: count, dtype: int64

How many rows matched?
5 rows matched between the tables

How many occurred only in the participant table? How many occurred only in the score table?
One observation occured in both the participant and score tables (separately).

In [15]:
participants["id"].is_unique
scores["id"].is_unique

True

In [17]:
participants["id"].duplicated().sum()
scores["id"].duplicated().sum()

np.int64(0)

Why should this be checked before a one-to-one join?
We want to assure that there is only one observation for each ID in the table.

In [21]:
validated = participants.merge(
    scores,
    on="id",
    how="left",
    validate="one_to_one"
)
duplicate_scores = pd.DataFrame({
    "id": [101, 101, 102],
    "score": [81, 83, 94]
})
participants.merge(
    duplicate_scores,
    on="id",
    validate="one_to_one"
)

MergeError: Merge keys are not unique in right dataset; not a one-to-one merge

Why is the error valuable?
It tells us duplicate IDs exist, making a 1:1 relationship impossible.

In [22]:
visits = pd.DataFrame({
    "id": [1, 1, 2],
    "visit": ["pre", "post", "pre"]
})

treatments = pd.DataFrame({
    "id": [1, 1, 2],
    "treatment": ["A", "B", "A"]
})
visits.merge(
    treatments,
    on="id"
)

,id,visit,treatment
0,1,pre,A
1,1,pre,B
2,1,post,A
3,1,post,B
4,2,pre,A


Why does ID 1 generate four rows?
Because there are two rows in each table for id 1, there are four combinations of those rows.

Was information duplicated?
Yes, information was duplicated twice for each row in the original tables.

Under what circumstances could this be correct?
For a purchase list, we might want to know if ID 1 purchased multiple things, including maybe shipping methods.

In [24]:
# Concatenate Rows
fall = pd.DataFrame({
    "id": [1, 2, 3],
    "semester": ["Fall"] * 3,
    "score": [80, 85, 90]
})

spring = pd.DataFrame({
    "id": [4, 5, 6],
    "semester": ["Spring"] * 3,
    "score": [78, 92, 88]
})
all_scores = pd.concat(
    [fall, spring],
    axis=0,
    ignore_index=True
)
all_scores

,id,semester,score
0,1,Fall,80
1,2,Fall,85
2,3,Fall,90
3,4,Spring,78
4,5,Spring,92
5,6,Spring,88


Why is concat() more natural than merge() here?
Because the tables contain the same variables, we want to stack them on top of each other instead of adding more variables through a merge.

In [25]:
fall = pd.DataFrame({
    "id": [1, 2],
    "score": [80, 85]
})

spring = pd.DataFrame({
    "id": [3, 4],
    "score": [90, 92],
    "campus": ["Flagstaff", "Flagstaff"]
})
pd.concat(
    [fall, spring],
    ignore_index=True
)

,id,score,campus
0,1,80,NaN
1,2,85,NaN
2,3,90,Flagstaff
3,4,92,Flagstaff


What does pandas do with the column that does not exist in fall?
It leaves N/As for the missing values in the other table.

In [26]:
# Wide Data
wide = pd.DataFrame({
    "id": [101, 102, 103, 104],
    "group": ["A", "B", "A", "B"],
    "score_pre": [72, 85, 79, 88],
    "score_post": [81, 91, 84, 90]
})
wide

,id,group,score_pre,score_post
0,101,A,72,81
1,102,B,85,91
2,103,A,79,84
3,104,B,88,90


How many rows represent each participant?
There is one row per participant.

How are repeated measurements represented?
Repeated measurements are represented in separate columns.

In [27]:
long = wide.melt(
    id_vars=["id", "group"],
    value_vars=["score_pre", "score_post"],
    var_name="time",
    value_name="score"
)
long

,id,group,time,score
0,101,A,score_pre,72
1,102,B,score_pre,85
2,103,A,score_pre,79
3,104,B,score_pre,88
4,101,A,score_post,81
5,102,B,score_post,91
6,103,A,score_post,84
7,104,B,score_post,90


How many rows now represent each participant?
Now there are two rows per participant.

What happened to the original score columns?
The columns were merged into one score column for both pre- and post-.

What does id_vars mean?
id_vars tells the melt function which columns we want to keep as they are, while repeating for values of the other variables.

What does value_vars mean?
value_vars shows which columns we want to turn into long format.

In [28]:
long["time"] = (
    long["time"]
    .str.replace("score_", "", regex=False)
)
long

,id,group,time,score
0,101,A,pre,72
1,102,B,pre,85
2,103,A,pre,79
3,104,B,pre,88
4,101,A,post,81
5,102,B,post,91
6,103,A,post,84
7,104,B,post,90


In [35]:
# Long to Wide
wide_again = long.pivot(
    index=["id", "group"],
    columns="time",
    values="score"
)
wide_again
wide_again = wide_again.reset_index()
wide_again

time,id,group,post,pre
0,101,A,81,72
1,102,B,91,85
2,103,A,84,79
3,104,B,90,88


Did you recover the same information as the original wide dataset?
Yes, the table is the same as the original wide table.

In [36]:
repeated = pd.DataFrame({
    "id": [1, 1, 1, 2, 2],
    "time": ["pre", "pre", "post", "pre", "post"],
    "score": [70, 74, 82, 80, 88]
})
repeated.pivot(
    index="id",
    columns="time",
    values="score"
)

ValueError: Index contains duplicate entries, cannot reshape

Why does this fail?
The pivot is attempting to move into a wide data set, but there is more than one value for the ID/Values for ID 1 and time pre.

Which assumption is violated?
It violates the 1:1 assumption for each combination of ID/Time.

In [37]:
summary = repeated.pivot_table(
    index="id",
    columns="time",
    values="score",
    aggfunc="mean"
)
summary

time,post,pre
id,,
1,82.0,72.0
2,88.0,80.0


Why should you investigate duplicate records before deciding that taking the mean is appropriate?
Since there are duplicate records, the averagfes are skewed from a single student testing more than once.

In [54]:
# Missing Data
clinical = pd.DataFrame({
    "id": [1, 2, 3, 4, 5, 6],
    "age": [24, 31, np.nan, 45, 28, 39],
    "group": ["A", "B", "A", "B", None, "A"],
    "score": [81, np.nan, 77, 92, 88, np.nan]
})
clinical
# clinical.isna()
# clinical.isna().sum()
# clinical.isna().mean() * 100

,id,age,group,score
0,1,24.0,A,81.0
1,2,31.0,B,NaN
2,3,NaN,A,77.0
3,4,45.0,B,92.0
4,5,28.0,None,88.0
5,6,39.0,A,NaN


In [45]:
# Identify Rows
clinical.loc[
    clinical["score"].isna()
]
clinical.loc[
    clinical["score"].notna()
]
clinical.loc[
    clinical[["age", "score"]]
    .isna()
    .any(axis=1)
]

,id,age,group,score
1,2,31.0,B,NaN
2,3,NaN,A,77.0
5,6,39.0,A,NaN


In [53]:
# Drop Missing Observations
complete_score = clinical.dropna(
    subset=["score"]
)
len(clinical)
len(complete_score)
complete_cases = clinical.dropna(
    subset=["age", "score"]
)
len(complete_cases)

3

How many observations were removed?
When checking for full scores, 2 observations were removed. However, when checking for all NAs, 3 rows were removed.

Has the analysis population changed?
It changes both the age representation as well as the groups represented.

What assumptions might make complete-case analysis problematic?
For a certain value of one variable, another variable may not be collected or is impossible to collect. Removing these rows could affect the analysis of the variable that is complete.

In [58]:
filled = clinical.copy()
filled["score"] = filled["score"].fillna(
    filled["score"].mean()
)
clinical["score"].describe()
filled["score"].describe()

count     6.000000
mean     84.500000
std       5.234501
min      77.000000
25%      81.875000
50%      84.500000
75%      87.125000
max      92.000000
Name: score, dtype: float64

Did the mean change?
Yes, the mean went down by 4.

Did the standard deviation change?
The standard deviation also reduced.

Why?
The standard deviation is the distance from the mean. If we insert some values AS the mean, then the distance calculated is bound to be shorter.

Is mean imputation automatically a good statistical strategy?
No, it doesn't substitute in for real, collected data.

In [61]:
# Missingness from Join
joined = participants.merge(
    scores,
    on="id",
    how="left"
)
joined.loc[
    joined["score"].isna()
]
joined

,id,age,group,score
0,101,24,A,81.0
1,102,31,B,94.0
2,103,27,A,88.0
3,104,19,B,NaN
4,105,42,A,91.0
5,106,35,B,85.0


Does a missing score here necessarily mean that the participant failed to provide a score? What other explanation exists?
No. It could be that the score was not recorded with the rest of the data, or that the sore was disqualified. The judges could've finish counting the best scores and tossed the other records before recording the other. There are several reasons data might be missing.

In [62]:
clinical["score_missing"] = (
    clinical["score"].isna()
)
clinical

,id,age,group,score,score_missing
0,1,24.0,A,81.0,False
1,2,31.0,B,NaN,True
2,3,NaN,A,77.0,False
3,4,45.0,B,92.0,False
4,5,28.0,None,88.0,False
5,6,39.0,A,NaN,True


In [63]:
pd.crosstab(
    clinical["group"],
    clinical["score_missing"]
)

score_missing,False,True
group,,
A,2,1
B,1,1


What question does this table help you investigate?
This table shows the number of missing values per group of another variable.

In [64]:
demographics = pd.DataFrame({
    "id": [101, 102, 103, 104, 105],
    "age": [24, 31, 27, 42, 36],
    "group": ["A", "B", "A", "B", "A"]
})
outcomes = pd.DataFrame({
    "id": [101, 102, 103, 105],
    "pre": [72, 80, 85, 78],
    "post": [81, 87, np.nan, 86]
})

In [72]:
# first, we'll join the groups
demoOut = demographics.merge(
    outcomes,
    on="id",
    how = "left",
    validate="one_to_one"
)

#melt the table into long format
analysis = demoOut.melt(
    id_vars=["id", "group"],
    value_vars=["pre", "post"],
    var_name="time",
    value_name="score"
)
analysis

,id,group,time,score
0,101,A,pre,72.0
1,102,B,pre,80.0
2,103,A,pre,85.0
3,104,B,pre,NaN
4,105,A,pre,78.0
5,101,A,post,81.0
6,102,B,post,87.0
7,103,A,post,NaN
8,104,B,post,NaN
9,105,A,post,86.0


In [ ]:
# Audit Missingness
analysis.isna().sum()
analysis.loc[
    analysis["score"].isna()
]

,id,group,time,score
3,104,B,pre,NaN
7,103,A,post,NaN
8,104,B,post,NaN


For each missing score, determine whether it arose because:

    an outcome record was absent entirely; or

    an outcome row existed but a particular measurement was missing.

These are different data problems.

Participant 104 was not present in the scores data set, so no outcomes were recoreded for them. Participant 103 had a pre score recorded, but did not have a post score recorded, accounting for the third missing row.

In [80]:
observed = analysis.dropna(
    subset=["score"]
)
n_before = len(analysis)
n_after = len(observed)
n_removed = n_before - n_after
n_removed

3

Why should this information appear in an analysis report?
This information is necessary to know how much of the data is complete, which gives a full view of how much analysis is valid.

In [81]:
analysis["id"].nunique()
demographics["id"].nunique()
analysis["time"].value_counts()
analysis.duplicated(
    subset=["id", "time"]
).sum()
assert (
    analysis["id"].nunique()
    == demographics["id"].nunique()
)

assert (
    analysis.duplicated(
        subset=["id", "time"]
    ).sum()
    == 0
)

In [82]:
# Part 30
jan = pd.DataFrame({
    "id": [1, 2],
    "month": ["Jan", "Jan"],
    "score": [80, 85]
})

feb = pd.DataFrame({
    "id": [1, 2],
    "month": ["Feb", "Feb"],
    "score": [82, 88]
})

mar = pd.DataFrame({
    "id": [1, 2],
    "month": ["Mar", "Mar"],
    "score": [84, np.nan]
})

In [ ]:
# Concatenate the three tables.
allMonths = pd.concat(
    [jan, feb, mar],
    axis=0,
    ignore_index=True
)

# Verify the number of rows.
len(allMonths)

# Identify missing scores.
allMonths.loc[
    allMonths["score"].isna()
]

# Pivot into wide form with one row per participant.
wideMonths = allMonths.pivot(
    index=["id"],
    columns="month",
    values="score"
).reset_index()
wideMonths

# Convert back to long form.
longMonth2 = wideMonths.melt(
    id_vars="id",
    value_vars=["Jan", "Feb", "Mar"],
    var_name="month",
    value_name="score"
)
longMonth2

# Verify that the structure is equivalent to the original combined data.
# Yes, the structure is the same as the concatenated data

,id,month,score
0,1,Jan,80.0
1,2,Jan,85.0
2,1,Feb,82.0
3,2,Feb,88.0
4,1,Mar,84.0
5,2,Mar,NaN


For each scenario, choose the most appropriate operation.
A) Two tables contain the same variables for different months.
Concat

B) One table contains demographics and another contains outcomes linked by participant ID.
Merge

C) Repeated measurements are stored in score_pre, score_post, and score_followup.
Melt

D)Long data should become one row per participant.
Pivot

E) You need every participant from the enrollment table, whether or not an outcome exists.
Left join

1. Merge
What is the conceptual difference between an inner join and a left join?
An inner join only keeps the keys that are in BOTH tables, while a left join will key all keys in the first table and remove the non-matching ones in the second table.

2. Keys
Why should you inspect duplicate keys before merging?
Duplicate keys can prevent merges from happening smoothly, especially when expecting a 1:1 merge.

3. Validation
What does validate="one_to_one" protect against?
One to one validation ensures there is one unique row for each key in both tables.
4. Concatenation
When is concat() more appropriate than merge()?
Concat is more appropriate when tables have the same variables but values we want to combine into a single table.

5. Reshaping
What is the difference between wide and long data?
Wide data keeps one row per key, but long data may have many rows per key for each observation.

6. Pivoting
Why might pivot() fail when pivot_table() succeeds?
Pivot_table better handles duplicates, while pivot will not accept them or treat them improperly.

7. Missingness
Why should missing values introduced by a join be interpreted differently from missing values already present in a source table?
Missing values from a join indicate that there may be duplicate rows for each key or one observation missing for one key while another exists, but a missing value in a source table indicates that value was not recorded to begin with.

8. Statistical practice
Why is dropping missing observations a statistical decision rather than only a programming operation?
Sometimes you need to analyise why rows are missing, including how those observations may not be possible. In some cases, when only caring about a variable with missing values, it is better to just drop anything not there.